In [ ]:
# Lab type: review
# Course: DS202 — Time Series Analysis & Forecasting
# Lesson: Trend, Seasonality, and Decomposition
# Task: The decompositions below are correctly implemented. Answer the judgment
#       questions about what they do and don't establish. Write 2-4 sentences each.

# Lab: Reading Decompositions Like a Practitioner

This lab runs three decompositions of the course dataset. All three execute
correctly — but they make different claims, and only some of those claims are
justified. Your job is to judge them.

**Outputs are cleared.** Run each cell to generate results.

## Setup

In [ ]:
!pip install pandas numpy scikit-learn statsmodels matplotlib --quiet

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
days = pd.date_range("2023-01-01", "2025-12-31", freq="D")
t = np.arange(len(days))

trend   = 200 + 0.15 * t
weekday = np.array([-14, -18, -11, -6, 9, 52, 61])[days.dayofweek]
yearly  = 38 * np.sin(2 * np.pi * (days.dayofyear - 320) / 365.25)
noise   = rng.normal(0, 16, len(days))

orders = pd.Series(trend + weekday + yearly + noise, index=days, name="orders").round()
print(f"{len(orders)} days, {orders.index[0].date()} to {orders.index[-1].date()}")
orders.head()

## Decomposition A: weekly period

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose
import matplotlib.pyplot as plt

result_a = seasonal_decompose(orders, model="additive", period=7)
fig = result_a.plot()
fig.set_size_inches(10, 7)
plt.tight_layout()
plt.show()

resid_a = result_a.resid.dropna()
print(f"residual std: {resid_a.std():.1f}")
print(f"residual lag-1 autocorr: {resid_a.autocorr(1):.3f}")
print(f"residual lag-7 autocorr: {resid_a.autocorr(7):.3f}")

**Question 1.** The trend panel of Decomposition A is not a straight line — it contains
a broad wave. We know the series has yearly seasonality. Is the wavy trend panel a
*defect* of this decomposition, a *wrong parameter choice*, or *expected behaviour*?
What exactly does "trend" mean in the output of `seasonal_decompose(period=7)`?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Question 1</summary>

Expected behaviour. With `period=7`, the trend is estimated by a centred 7-day moving
average, so "trend" means "everything slower than 7 days" — which includes the yearly
cycle. Classical decomposition extracts exactly one seasonality: the one whose period
you pass. If you need the yearly component isolated, decompose again at a coarser
grain (weekly data, `period=52`) or use a multi-seasonal method (MSTL). Nothing here
is broken; the label "trend" is just narrower than it sounds.

</details>

## Decomposition B: a different period claim

In [ ]:
result_b = seasonal_decompose(orders, model="additive", period=30)

resid_b = result_b.resid.dropna()
print(f"period=30 residual std: {resid_b.std():.1f}   lag-7 autocorr: {resid_b.autocorr(7):.3f}")
print(f"period=7  residual std: {resid_a.std():.1f}   lag-7 autocorr: {resid_a.autocorr(7):.3f}")

one_cycle = result_b.seasonal.iloc[:30]
print("\nfirst 'monthly' seasonal cycle (should it look this regular?):")
print(one_cycle.round(1).to_string())

**Question 2.** Decomposition B ran without error and produced a confident-looking
30-day seasonal component. Using the residual diagnostics printed above, make the case
that `period=30` is the wrong claim for this series. Why doesn't `seasonal_decompose`
warn you?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Question 2</summary>

The `period=30` residual still has strong lag-7 autocorrelation — the weekly pattern
survived into "unexplained noise", which is exactly what a correct decomposition must
not leave behind. Its residual std is also visibly larger than A's. The 30-day
"seasonal" shape is mostly an averaging artefact of the true 7-day cycle beating
against a 30-day window. `seasonal_decompose` can't warn you because the period is
*your* claim about the data — the function just averages by position-in-cycle for
whatever period you assert. The residual diagnostics are the warning system.

</details>

## Section 3: A tempting feature

In [ ]:
# A colleague proposes this feature for a live daily forecasting model:
proposed_feature = result_a.trend          # "the de-noised level of the series"
print(proposed_feature["2025-12-20":"2025-12-31"])

**Question 3.** Look at the last few values of the proposed feature. Explain (a) why
the final 3 days are NaN, (b) what that reveals about how the trend is computed, and
(c) why this feature can therefore never be used by a live forecasting model — even for
the historical rows where it *isn't* NaN.

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Question 3</summary>

(a) The trend is a *centred* 7-day moving average: day *t* needs days *t−3 … t+3*, and
the last 3 days of the series have no *t+3* yet, so they're NaN. (b) That NaN pattern is
the proof that every non-NaN trend value was computed using three days of *future*
data. (c) A live model predicting day *t* would need a feature computable at *t* — this
one never is ("today's" value is always among the NaNs). Training on the historical
non-NaN rows teaches the model to rely on future-peeking information that will not
exist at prediction time: temporal leakage, Lesson 3's centred-window rule exactly.

</details>

**Question 4 (composition).** Our synthetic series was built additively. Suppose instead
the store's weekend *lift* was proportional — weekends run ~20% above the weekday level,
whatever that level is. Which single change would you make to the `seasonal_decompose`
call, and what would you look for in the raw plot and in the residuals to confirm the
choice?

*(Write your answer here.)*

<details>
<summary>🔑 Reveal answer — Question 4</summary>

Use `model="multiplicative"`. Confirmation: in the raw plot, seasonal amplitude that
widens as the level grows (a funnel) indicates proportional seasonality; after
decomposing, the residuals of the *correct* model show no level-dependence, while the
wrong (additive) model leaves residuals whose spread grows with the trend. When unsure,
run both and compare residual structure — the better composition leaves less behind.

</details>

## Summary

> **Complete each sentence in one line.**

1. In `seasonal_decompose(period=7)`, "trend" actually means ________.
2. A wrong `period` fails silently; the warning system is ________.
3. The centred trend can't be a live feature because ________.

<details>
<summary>🔑 Reveal summary answers</summary>

1. "Trend" means **everything slower than the stated period** — including any longer
   seasonality, like the yearly wave.
2. The warning system is **the residual: remaining autocorrelation (especially at the
   true seasonal lag) and inflated residual spread**.
3. Because **it's computed from a centred window that includes future days — the
   trailing NaNs at the series' end prove it can never be computed for "today"**.

</details>